# MNIST TensorFlow Notebook

This notebook uses the Python kernel configured for this workspace.

In [1]:
import tensorflow as tf
import numpy as np
print('TensorFlow', tf.__version__)
print('NumPy', np.__version__)

TensorFlow 2.21.0
NumPy 2.5.1


In [2]:
from __future__ import absolute_import, division, print_function

import tensorflow as tf
import numpy as np

# MNIST dataset parameters
num_classes = 10  # total classes (0-9 digits)
num_features = 784  # data features in our input (28*28 = 784)

# Training parameters
learning_rate = 0.001
training_steps = 3000
batch_size = 256
display_step = 100

# Neural Network parameters
n_hidden_1 = 128  # 1st layer number of neurons
n_hidden_2 = 256  # 2nd layer number of neurons

# Prepare MNIST data.
# download the dataset
from tensorflow.keras.datasets import mnist

(x_train, y_train), (x_test, y_test) = mnist.load_data()

# Convert to float32
x_train, x_test = np.array(x_train, np.float32), np.array(x_test, np.float32)

# Flatten images to 1-D vector of 784 features (28*28)
x_train, x_test = x_train.reshape([-1, num_features]), x_test.reshape([-1, num_features])

# Normalize image values from [0, 255] ---> [0, 1]
x_train, x_test = x_train / 255., x_test / 255.

# Use tf.data API to shuffle and batch data
train_data = tf.data.Dataset.from_tensor_slices((x_train, y_train))
train_data = train_data.repeat().shuffle(5000).batch(batch_size).prefetch(1)

# store layers weights and biases
# A random number generator to initialize weights
random_normal = tf.initializers.RandomNormal()

weights = {
    "h1": tf.Variable(random_normal([num_features, n_hidden_1])),
    "h2": tf.Variable(random_normal([n_hidden_1, n_hidden_2])),
    "out": tf.Variable(random_normal([n_hidden_2, num_classes]))
}

biases = {
    "b1": tf.Variable(random_normal([n_hidden_1])),
    "b2": tf.Variable(random_normal([n_hidden_2])),
    "out": tf.Variable(random_normal([num_classes]))
}

print("Shape W_1: ", weights["h1"].shape)
print("Shape W_2: ", weights["h2"].shape)
print("Shape W_Out: ", weights["out"].shape)

print("Shape B_1: ", biases["b1"].shape)
print("Shape B_2: ", biases["b2"].shape)
print("Shape B_Out: ", biases["out"].shape)


# Create model
def neural_net(x):
    # Hidden fully connected layer with 128 neurons.
    layer_1 = tf.add(tf.matmul(x, weights['h1']), biases['b1'])  # Z1
    # Apply sigmoid to layer_1 output for non-linearity
    layer_1 = tf.nn.sigmoid(layer_1)  # A1

    # Hidden fully connected layer with 256 neurons.
    layer_2 = tf.add(tf.matmul(layer_1, weights['h2']), biases['b2'])  # Z2

    # apply sigmoid to layer_2 output
    layer_2 = tf.nn.sigmoid(layer_2)  # A2

    # Output fully connected layer with a neuron for each class ---> 10
    out_layer = tf.matmul(layer_2, weights['out']) + biases['out']  # Z3

    # Apply softmax to normalize the logits to a probability distribution
    return tf.nn.softmax(out_layer)


# Cross-Entropy loss function
def cross_entropy(y_pred, y_true):
    # Encode label to a one hot vector
    y_true = tf.one_hot(y_true, depth=num_classes)
    # Clip predictions values to avoid log(0) error.
    y_pred = tf.clip_by_value(y_pred, 1e-9, 1.)
    # Compute cross-entropy loss
    return tf.reduce_mean(-tf.reduce_sum(y_true * tf.math.log(y_pred)))


# Accuracy metrics
def accuracy(y_pred, y_true):
    # Predicted class is the index of highest score in prediction vector( ie argmax)
    correct_prediction = tf.equal(tf.argmax(y_pred, 1), tf.cast(y_true, tf.int64))
    return tf.reduce_mean(tf.cast(correct_prediction, tf.float32), 1)


# Stochastic gradient descent optimizer
optimizer = tf.optimizers.SGD(learning_rate)


# Optimizing process
def run_optimization(x, y):
    # Wrap computation inside a GradientTape for automatic differentiation
    with tf.GradientTape() as g:
        pred = neural_net(x)
        loss = cross_entropy(pred, y)

    # Variables to update, i.e. trainable variables.
    trainable_variables = list(weights.values()) + list(biases.values())

    # Compute gradients
    gradients = g.gradient(loss, trainable_variables)

    # Update W and b following gradients
    optimizer.apply_gradients(zip(gradients, trainable_variables))

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step
Shape W_1:  (784, 128)
Shape W_2:  (128, 256)
Shape W_Out:  (256, 10)
Shape B_1:  (128,)
Shape B_2:  (256,)
Shape B_Out:  (10,)
